# Figure 2: Activity-based evaluation with SLAY

**Paper:** BATTLE-AMP

**Per-panel stories:**
- **(a)** Models that appear effective at binary AMP/non-AMP classification lose most of their discriminative power when evaluated on experimentally confirmed antimicrobial activity.
- **(b)** Most classifiers produce unacceptably high false-positive rates on GeneralActivity despite appearing specific on AMP/non-AMP.
- **(c)** Under realistic screening conditions (SLAY, 1.8% positives), nearly all models lose their ability to enrich for true actives, with only highly selective regressors maintaining meaningful precision.

**Layout:** 3 rows. a (full width MCC), b (full width FPR), c (50/50 Precision@100: GA vs SLAY, same 0-1 y-axis).

**Sorting:** Best-performing model on the left in every bar panel.

**Moved to supplement:** AUROC vs pAUROC(FPR<0.01) scatter (former panel b).


In [5]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# Paths
# ============================================================
CLF_FILE = Path( "../results/aggregated/classification_results.tsv")
FIGURE_DIR = Path("../figures")
FIGURE_DIR.mkdir(exist_ok=True)
# ============================================================
# Figure constants (Briefings in Bioinformatics)
# ============================================================
PANEL_W = 6.5       # max width
TARGET_DPI = 600

CLF_COLOR  = "#d95f6e"   # muted rose      -- classifiers
ACT_COLOR  = "#9467bd"   # muted purple    -- HydrAMP-MIC
REG_COLOR  = "#8c8c8c"   # medium grey     -- regressors

GRID_COLOR = "#dddddd"
FONTSIZE_TICK  = 6
FONTSIZE_LABEL = 7
FONTSIZE_PANEL = 10

# ============================================================
# Models
# ============================================================
CLASSIFIERS = [
    "hydramp-amp-classifier", "ampscanner", "amplify",
    "sensexamp-classifier", "ampeppy", "ampredmfa", "mole-amp",
]
ACTIVITY_AWARE = ["hydramp-mic-classifier"]
REGRESSORS = [
    "mbc-attention", "ampredictor",
    "sensexamp-ecoli", "sensexamp-saureus",
    "deep-amp-cnn-gramneg", "deep-amp-cnn-grampos",
    "deep-amp-lstm-gramneg", "deep-amp-lstm-grampos",
    "apex-ecoli", "apex-saureus", "apex-min",
    "apex-abaumannii", "apex-paeruginosa", "apex-kpneumoniae",
]
ALL_MODELS = CLASSIFIERS + ACTIVITY_AWARE + REGRESSORS

MODEL_DISPLAY = {
    "hydramp-amp-classifier": "HydrAMP$_{AMP}$",
    "ampscanner": "AMP Scanner$_2$",
    "amplify": "AMPlify",
    "sensexamp-classifier": "sAMPpred$_{clf}$",
    "ampeppy": "amPEPpy",
    "ampredmfa": "AMPpred-MFA",
    "mole-amp": "MoLE-AMP",
    "hydramp-mic-classifier": "HydrAMP$_{MIC}$",
    "mbc-attention": "MBC-Attention",
    "ampredictor": "AMPredictor",
    "sensexamp-ecoli": "sAMPpred$_{EC}$",
    "sensexamp-saureus": "sAMPpred$_{SA}$",
    "deep-amp-cnn-gramneg": "Deep-AMP$_{CNN-}$",
    "deep-amp-cnn-grampos": "Deep-AMP$_{CNN+}$",
    "deep-amp-lstm-gramneg": "Deep-AMP$_{LSTM-}$",
    "deep-amp-lstm-grampos": "Deep-AMP$_{LSTM+}$",
    "apex-ecoli": "APEX$_{EC}$",
    "apex-saureus": "APEX$_{SA}$",
    "apex-min": "APEX$_{min}$",
    "apex-abaumannii": "APEX$_{AB}$",
    "apex-paeruginosa": "APEX$_{PA}$",
    "apex-kpneumoniae": "APEX$_{KP}$",
}


def model_color(m):
    if m in CLASSIFIERS:
        return CLF_COLOR
    elif m in ACTIVITY_AWARE:
        return ACT_COLOR
    return REG_COLOR


def short_name(m):
    return MODEL_DISPLAY.get(m, m)


# ============================================================
# Load data
# ============================================================
df = pd.read_csv(CLF_FILE, sep="\t")
df = df[df["variant"] != "example-model"].copy()

amp = df[df["task"] == "amp"].set_index("variant")
ga  = df[df["task"] == "broad_activity"].set_index("variant")

ga_models = [m for m in ga.index if m in ALL_MODELS]

ga_base_rate = ga.iloc[0]["n_positive"] / (
    ga.iloc[0]["n_positive"] + ga.iloc[0]["n_negative"]
)

slay_df = df[df["task"] == "slay"].set_index("variant")
slay_models = [m for m in ga_models if m in slay_df.index]
slay_base_rate = slay_df.iloc[0]["n_positive"] / (
    slay_df.iloc[0]["n_positive"] + slay_df.iloc[0]["n_negative"]
)

# ============================================================
# Per-panel model orderings (best on left)
# ============================================================

def safe_val(frame, m, col, default=np.nan):
    return frame.loc[m, col] if m in frame.index else default

# Panel a: sorted by GA MCC descending
order_a = sorted(ga_models,
    key=lambda m: safe_val(ga, m, "mcc", -1), reverse=True)

# Panel b: sorted by GA FPR ascending (lowest = best)
order_b = sorted(ga_models,
    key=lambda m: safe_val(ga, m, "fpr", 2))

# Panel c-left: sorted by GA Precision@100 descending
order_c1 = sorted(ga_models,
    key=lambda m: safe_val(ga, m, "precision_at_k", -1), reverse=True)

# Panel c-right: sorted by SLAY Precision@100 descending
order_c2 = sorted(slay_models,
    key=lambda m: safe_val(slay_df, m, "precision_at_k", -1), reverse=True)

n_a, n_b = len(order_a), len(order_b)
n_c1, n_c2 = len(order_c1), len(order_c2)

print(f"Models: a={n_a}, b={n_b}, c1={n_c1}, c2={n_c2}")
print(f"GA base rate: {ga_base_rate:.3f}")
print(f"SLAY base rate: {slay_base_rate:.4f} ({slay_base_rate*100:.2f}%)")

# ============================================================
# Global style
# ============================================================
matplotlib.rcParams["font.family"] = "sans-serif"
matplotlib.rcParams["font.sans-serif"] = [
    "Helvetica", "Arial", "Liberation Sans", "DejaVu Sans",
]
matplotlib.rcParams["mathtext.default"] = "regular"
matplotlib.rcParams["axes.linewidth"] = 0.5
matplotlib.rcParams["xtick.major.width"] = 0.4
matplotlib.rcParams["ytick.major.width"] = 0.4


def style_ax(ax, ylabel=None, xlabel=None):
    ax.set_facecolor("white")
    ax.yaxis.grid(True, color=GRID_COLOR, linewidth=0.4)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=FONTSIZE_TICK, length=2)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=FONTSIZE_LABEL)
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=FONTSIZE_LABEL)


def set_model_xlabels(ax, order, rotation=55, fontsize=6):
    ax.set_xticks(np.arange(len(order)))
    labels = ax.set_xticklabels(
        [short_name(m) for m in order],
        rotation=rotation, ha="right", fontsize=fontsize,
    )
    for lbl, m in zip(labels, order):
        lbl.set_color(model_color(m))
    ax.set_xlim(-0.6, len(order) - 0.4)


def type_legend(ax, loc="upper right", ncol=3):
    handles = [
        Patch(facecolor=CLF_COLOR, edgecolor="none", label="Classifier"),
        Patch(facecolor=ACT_COLOR, edgecolor="none", label="HydrAMP$_{MIC}$"),
        Patch(facecolor=REG_COLOR, edgecolor="none", label="Regressor"),
    ]
    ax.legend(handles=handles, fontsize=6, loc=loc, frameon=True,
              facecolor="white", edgecolor="#cccccc", ncol=ncol,
              handlelength=1.0, handleheight=0.7, columnspacing=0.8)


def task_legend(ax, loc="upper right"):
    ax.legend(
        [Patch(facecolor="#888888", alpha=0.35, edgecolor="#aaaaaa"),
         Patch(facecolor="#888888", alpha=1.0, edgecolor="#aaaaaa")],
        ["AMP/non-AMP", "GeneralActivity"],
        fontsize=6, loc=loc, frameon=True,
        facecolor="white", edgecolor="#cccccc",
        handlelength=1.0, handleheight=0.7,
    )




Models: a=13, b=13, c1=13, c2=13
GA base rate: 0.784
SLAY base rate: 0.0178 (1.78%)


In [9]:
# ============================================================
# Build figure: 3 rows, clean
# ============================================================
fig = plt.figure(figsize=(PANEL_W, 7.0), dpi=TARGET_DPI, facecolor="white")

gs = gridspec.GridSpec(
    3, 1, hspace=0.48,
    height_ratios=[1.0, 1.0, 1.0],
    left=0.10, right=0.97, top=0.975, bottom=0.06,
)

ax_a = fig.add_subplot(gs[0])
ax_b = fig.add_subplot(gs[1])

# Panel c: 50/50 split
gs_c = gridspec.GridSpecFromSubplotSpec(
    1, 2, subplot_spec=gs[2], wspace=0.25, width_ratios=[1, 1],
)
ax_c1 = fig.add_subplot(gs_c[0])
ax_c2 = fig.add_subplot(gs_c[1])


# ================================================================
# Panel A: MCC grouped bars (sorted by GA MCC descending)
# ================================================================
xa = np.arange(n_a)
mcc_amp_a = [safe_val(amp, m, "mcc") for m in order_a]
mcc_ga_a  = [safe_val(ga,  m, "mcc") for m in order_a]

bar_w = 0.38
ax_a.bar(xa - bar_w / 2, mcc_amp_a, bar_w,
         color=[model_color(m) for m in order_a],
         alpha=0.35, edgecolor="#aaaaaa", linewidth=0.3)
ax_a.bar(xa + bar_w / 2, mcc_ga_a, bar_w,
         color=[model_color(m) for m in order_a],
         alpha=1.0, edgecolor="#aaaaaa", linewidth=0.3)

set_model_xlabels(ax_a, order_a)
ax_a.set_ylim(0, 0.88)
style_ax(ax_a, ylabel="MCC")
task_legend(ax_a, loc="upper right")
type_legend(ax_a, loc="upper center", ncol=3)


# ================================================================
# Panel B: FPR bars (sorted by GA FPR ascending)
# ================================================================
xb = np.arange(n_b)
fpr_amp_b = [safe_val(amp, m, "fpr") for m in order_b]
fpr_ga_b  = [safe_val(ga,  m, "fpr") for m in order_b]

bar_w = 0.38
ax_b.bar(xb - bar_w / 2, fpr_amp_b, bar_w,
         color=[model_color(m) for m in order_b],
         alpha=0.35, edgecolor="#aaaaaa", linewidth=0.3)
ax_b.bar(xb + bar_w / 2, fpr_ga_b, bar_w,
         color=[model_color(m) for m in order_b],
         alpha=1.0, edgecolor="#aaaaaa", linewidth=0.3)

set_model_xlabels(ax_b, order_b)
ax_b.set_ylim(0, 1.05)
style_ax(ax_b, ylabel="False Positive Rate")
task_legend(ax_b, loc="upper left")


# ================================================================
# Panel C: Precision@100 (GA left, SLAY right, SAME y-axis 0-1)
# ================================================================

# -- C1: GeneralActivity --
xc1 = np.arange(n_c1)
prec_ga = [safe_val(ga, m, "precision_at_k") for m in order_c1]

ax_c1.bar(xc1, prec_ga, 0.65,
          color=[model_color(m) for m in order_c1],
          edgecolor="#aaaaaa", linewidth=0.3)

# Positive fraction reference (dashed, explained in caption)
ax_c1.axhline(ga_base_rate, color="#aaaaaa", linewidth=0.5,
              linestyle="--", zorder=0)

set_model_xlabels(ax_c1, order_c1)
ax_c1.set_ylim(0, 1.05)
style_ax(ax_c1, ylabel="Precision@100")
ax_c1.set_title("GeneralActivity", fontsize=7, pad=3, color="#555555")


# -- C2: SLAY --
xc2 = np.arange(n_c2)
prec_slay = [safe_val(slay_df, m, "precision_at_k") for m in order_c2]

ax_c2.bar(xc2, prec_slay, 0.65,
          color=[model_color(m) for m in order_c2],
          edgecolor="#aaaaaa", linewidth=0.3)

# Positive fraction reference (dashed, explained in caption)
ax_c2.axhline(slay_base_rate, color="#aaaaaa", linewidth=0.5,
              linestyle="--", zorder=0)

set_model_xlabels(ax_c2, order_c2)
ax_c2.set_ylim(0, 1.05)   # SAME scale as GA panel
style_ax(ax_c2, ylabel="Precision@100")
ax_c2.set_title("SLAY (438k sequences, 1.8% positive)",
                fontsize=7, pad=3, color="#555555")


# ================================================================
# Panel labels
# ================================================================
for label, ax in [("a", ax_a), ("b", ax_b), ("c", ax_c1)]:
    pos = ax.get_position()
    fig.text(pos.x0 - 0.07, pos.y1 + 0.005, label,
             fontsize=FONTSIZE_PANEL, fontweight="bold",
             va="bottom", ha="left")


# ================================================================
# Save
# ================================================================
out_pdf = FIGURE_DIR / "figure2_activity_slay.pdf"
out_png = FIGURE_DIR / "figure2_activity_slay.png"
fig.savefig(str(out_pdf), bbox_inches="tight", pad_inches=0.02,
            dpi=TARGET_DPI, facecolor="white")
fig.savefig(str(out_png), bbox_inches="tight", pad_inches=0.02,
            dpi=TARGET_DPI, facecolor="white")
print(f"\nSaved {out_pdf} and {out_png}")
plt.close(fig)


# ================================================================
# Print SLAY summary table
# ================================================================
print("\n" + "="*72)
print("SLAY RESULTS SUMMARY (for manuscript text / Table S8)")
print("="*72)
print(f"{'Model':<25} {'Type':<12} {'TPR%':>6} {'FPR%':>6} "
      f"{'P@100':>6} {'LR+':>7}")
print("-"*72)
for m in order_c2:
    row = slay_df.loc[m]
    tpr = row["tpr"] * 100
    fpr = row["fpr"] * 100
    p100 = row["precision_at_k"]
    lr_plus = tpr / fpr if fpr > 0 else float("inf")
    mtype = ("Classifier" if m in CLASSIFIERS
             else "Activity-aware" if m in ACTIVITY_AWARE
             else "Regressor")
    print(f"{short_name(m):<25} {mtype:<12} {tpr:>6.2f} {fpr:>6.2f} "
          f"{p100:>6.2f} {lr_plus:>7.2f}")
print("-"*72)
print(f"Positive fraction: {slay_base_rate:.4f} "
      f"({int(slay_df.iloc[0]['n_positive'])} / "
      f"{int(slay_df.iloc[0]['n_positive'] + slay_df.iloc[0]['n_negative'])})")



Saved ../figures/figure2_activity_slay.pdf and ../figures/figure2_activity_slay.png

SLAY RESULTS SUMMARY (for manuscript text / Table S8)
Model                     Type           TPR%   FPR%  P@100     LR+
------------------------------------------------------------------------
APEX$_{PA}$               Regressor      0.79   0.03   0.45   22.96
APEX$_{SA}$               Regressor      0.77   0.03   0.44   25.38
APEX$_{EC}$               Regressor      0.84   0.04   0.40   19.85
APEX$_{KP}$               Regressor      0.78   0.03   0.40   24.55
APEX$_{AB}$               Regressor      0.89   0.04   0.32   19.74
APEX$_{min}$              Regressor      2.26   0.84   0.29    2.70
HydrAMP$_{MIC}$           Activity-aware   4.67   2.08   0.10    2.24
AMPlify                   Classifier    25.52  22.52   0.05    1.13
HydrAMP$_{AMP}$           Classifier    34.86  31.45   0.04    1.11
MBC-Attention             Regressor     32.29  30.09   0.04    1.07
amPEPpy                   Classifier 

## Alt text

**Figure 2.** Activity-based benchmarking of AMP prediction models across three
evaluation regimes. Models color-coded by type: classifiers (rose), HydrAMP-MIC
(purple), regressors (grey). All bar panels sorted with best performer on the left.
**(a)** Grouped bar chart of MCC, sorted by GeneralActivity MCC descending. Light
bars: AMP/non-AMP; full-opacity bars: GeneralActivity. Classifiers suffer large
drops moving to GeneralActivity; regressors and HydrAMP-MIC maintain or improve.
**(b)** False positive rate on AMP/non-AMP (light) and GeneralActivity (full),
sorted by GeneralActivity FPR ascending. APEX species regressors remain near zero;
most classifiers exceed 0.78 on GeneralActivity.
**(c)** Precision@100 with identical 0-1 y-axis on both halves, each sorted
independently. Left: GeneralActivity (dashed line marks positive fraction 0.78),
where most models achieve near-perfect precision. Right: SLAY dataset (438,484
random sequences from Tucker et al., 1.8% positive, dashed line marks positive
fraction 0.02), where APEX species regressors maintain 40-45% precision,
HydrAMP-MIC reaches 10%, and most classifiers collapse to near 2%.
